In [5]:
:opt 3
:dep kolmox = { path = "../src/lib" }
:dep plotly = "0.13"
:dep .

use benchmark::{BenchmarkResult, benchmarks};
use kolmox::compress::{
    brotli::CompressBrotli,
    zstd::CompressZstd,
    Compressor,
    cache::{Cache, InMemoryCache, NoCache},
};
use std::time::{Duration, Instant};

fn read_from_file(file_path: &str) -> String {
    std::fs::read_to_string(file_path).expect("Failed to read file")
}

Error: No such file or directory (os error 2): "../src/lib"

In [ ]:
const FILE_PATH: &str =
    "../../dataset/imdb/list/ls541382956/?ref_=tt_urls_2.html";

println!("Brotli Compression Benchmark");
let page_html = read_from_file(FILE_PATH);
let mut results = Vec::new();

for quality in 3..11 {
    for lg_window_size in 20..=22 {
        let start = Instant::now();
        let compressor = CompressBrotli::<InMemoryCache>::new(quality, lg_window_size);
        let result = compressor.get_distance(&page_html, &page_html);
        let duration = start.elapsed();

        let benchmark_result = benchmark::BenchmarkResult {
            quality,
            lg_window_size,
            compression_ratio: result,
            duration,
        };

        results.push(benchmark_result.clone());
        println!("Quality: {quality}, LG Window Size: {lg_window_size}, Compression Ratio: {:.6}, Time: {:?}", result, duration);
    }
}

let plot = benchmark::point_series(&results);
plot

Brotli Compression Benchmark
Quality: 3, LG Window Size: 20, Compression Ratio: 0.577553, Time: 33.793875ms
Quality: 3, LG Window Size: 21, Compression Ratio: 0.000223, Time: 27.4525ms
Quality: 3, LG Window Size: 22, Compression Ratio: 0.000223, Time: 27.890875ms
Quality: 4, LG Window Size: 20, Compression Ratio: 0.540567, Time: 43.867042ms
Quality: 4, LG Window Size: 21, Compression Ratio: 0.000053, Time: 37.652916ms
Quality: 4, LG Window Size: 22, Compression Ratio: 0.000053, Time: 32.059792ms
Quality: 5, LG Window Size: 20, Compression Ratio: 0.552726, Time: 75.320291ms
Quality: 5, LG Window Size: 21, Compression Ratio: 0.000080, Time: 56.0955ms
Quality: 5, LG Window Size: 22, Compression Ratio: 0.000080, Time: 55.359417ms
Quality: 6, LG Window Size: 20, Compression Ratio: 0.547157, Time: 80.637667ms
Quality: 6, LG Window Size: 21, Compression Ratio: 0.000081, Time: 61.885208ms
Quality: 6, LG Window Size: 22, Compression Ratio: 0.000081, Time: 60.87075ms
Quality: 7, LG Window Size: 

In [ ]:
let zstd = CompressZstd::<InMemoryCache>::recommended();

In [ ]:
benchmarks::distance_matrix::heatmap(&zstd, "euronews.com")

In [ ]:
benchmarks::distance_matrix::heatmap(&zstd, "amazon")

In [ ]:
benchmarks::distance_matrix::heatmap(&zstd, "imdb")

In [ ]:
benchmarks::distance_matrix::heatmap(&zstd, "wikipedia")

In [ ]:
let (labels_wiki, labels_grok, matrix) : (Vec<String>, Vec<String>, Vec<Vec<f64>>) = benchmarks::wiki_vs_grok::compute_distance_matrix("grokvswiki", &zstd);

In [ ]:
benchmarks::wiki_vs_grok::heatmap(&labels_wiki, &labels_grok, &matrix)

In [ ]:
benchmarks::wiki_vs_grok::histogram(&matrix)

Let's compare two very similar pages: one from [ru.ruwiki.ru](https://ru.ruwiki.ru/wiki/%D0%9A%D0%BE%D0%BB%D0%BC%D0%BE%D0%B3%D0%BE%D1%80%D0%BE%D0%B2,_%D0%90%D0%BD%D0%B4%D1%80%D0%B5%D0%B9_%D0%9D%D0%B8%D0%BA%D0%BE%D0%BB%D0%B0%D0%B5%D0%B2%D0%B8%D1%87) (a ru.wikipedia.org clone with a fashionable design and unnecessary AI features), and another from [ru.wikipedia.org](https://ru.wikipedia.org/wiki/%D0%9A%D0%BE%D0%BB%D0%BC%D0%BE%D0%B3%D0%BE%D1%80%D0%BE%D0%B2,_%D0%90%D0%BD%D0%B4%D1%80%D0%B5%D0%B9_%D0%9D%D0%B8%D0%BA%D0%BE%D0%BB%D0%B0%D0%B5%D0%B2%D0%B8%D1%87).

In [ ]:
let page_wiki = read_from_file(
    "assets/wikipedia.txt",
);
let page_ruwiki = read_from_file(
    "assets/ruwiki.txt",
);
let zstd = CompressZstd::<NoCache>::recommended();
zstd.get_distance(&page_wiki, &page_ruwiki)

0.06940311078668089